# E-commerce Sales Analytics Dashboard**Data Science Task # 09 — Gexton Education Summer Internship Program**Dataset: [Global E-Commerce Sales & Customer Data](https://www.kaggle.com/datasets/muhammadaammartufail/global-e-commerce-sales-and-customer-data) (Kaggle)This notebook covers data cleaning, exploratory data analysis, and machine learning model development to predict e-commerce sales.

## 1. Import Libraries

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import OneHotEncoder, StandardScalerfrom sklearn.compose import ColumnTransformerfrom sklearn.pipeline import Pipelinefrom sklearn.linear_model import LinearRegressionfrom sklearn.tree import DecisionTreeRegressorfrom sklearn.ensemble import RandomForestRegressorfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_scoreimport picklesns.set_style('whitegrid')plt.rcParams['figure.dpi'] = 100

## 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv('global_ecommerce_sales.csv')print(df.shape)df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Data CleaningCheck for missing values and duplicate rows, and convert `Order_Date` to a proper datetime type.

In [ ]:
print('Missing values:\n', df.isnull().sum())print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])df = df.drop_duplicates()df = df.dropna()print('Cleaned shape:', df.shape)

## 4. Feature EngineeringExtract date-based features and derive profit margin for deeper analysis.

In [ ]:
df['Year'] = df['Order_Date'].dt.yeardf['Month'] = df['Order_Date'].dt.monthdf['MonthName'] = df['Order_Date'].dt.strftime('%b %Y')df['DayOfWeek'] = df['Order_Date'].dt.day_name()df['Quarter'] = df['Order_Date'].dt.quarterdf['Profit_Margin_Pct'] = (df['Profit'] / df['Total_Sales'] * 100).round(2)df.head()

In [ ]:
# Save cleaned datasetdf.to_csv('global_ecommerce_sales_cleaned.csv', index=False)print('Cleaned dataset saved.')

## 5. Exploratory Data Analysis (EDA)

### 5.1 Monthly Sales Trend

In [ ]:
monthly = df.groupby(df['Order_Date'].dt.to_period('M')).agg(    Total_Sales=('Total_Sales','sum'), Orders=('Order_ID','count')).reset_index()monthly['Order_Date'] = monthly['Order_Date'].dt.to_timestamp()plt.figure(figsize=(11,4.5))plt.plot(monthly['Order_Date'], monthly['Total_Sales'], marker='o', color='#2a6f97', linewidth=2)plt.title('Monthly Sales Trend', fontsize=14, fontweight='bold')plt.xlabel('Month'); plt.ylabel('Total Sales ($)')plt.xticks(rotation=45)plt.tight_layout()plt.show()

### 5.2 Top Selling Products

In [ ]:
top_products = df.groupby('Product_Name')['Total_Sales'].sum().sort_values(ascending=False).head(10)plt.figure(figsize=(10,5))sns.barplot(x=top_products.values, y=top_products.index, hue=top_products.index, palette='viridis', legend=False)plt.title('Top 10 Selling Products by Revenue', fontsize=14, fontweight='bold')plt.xlabel('Total Sales ($)'); plt.ylabel('')plt.tight_layout()plt.show()

### 5.3 Best Performing Categories

In [ ]:
cat_perf = df.groupby('Product_Category').agg(Total_Sales=('Total_Sales','sum'), Profit=('Profit','sum')).sort_values('Total_Sales', ascending=False)cat_perf

In [ ]:
x = np.arange(len(cat_perf)); width = 0.38plt.figure(figsize=(9,5))plt.bar(x-width/2, cat_perf['Total_Sales'], width, label='Total Sales', color='#2a6f97')plt.bar(x+width/2, cat_perf['Profit'], width, label='Profit', color='#f4a261')plt.xticks(x, cat_perf.index, rotation=15)plt.title('Category Performance: Sales vs Profit', fontsize=14, fontweight='bold')plt.legend(); plt.tight_layout(); plt.show()

### 5.4 Regional Sales Performance

In [ ]:
region_perf = df.groupby('Region')['Total_Sales'].sum().sort_values(ascending=False)plt.figure(figsize=(8,5))sns.barplot(x=region_perf.index, y=region_perf.values, hue=region_perf.index, palette='crest', legend=False)plt.title('Regional Sales Performance', fontsize=14, fontweight='bold')plt.ylabel('Total Sales ($)'); plt.xlabel('')plt.xticks(rotation=20)plt.tight_layout(); plt.show()

### 5.5 Customer Purchase Behavior

In [ ]:
seg = df.groupby('Customer_Segment').agg(Total_Sales=('Total_Sales','sum'), Avg_Order=('Total_Sales','mean'), Orders=('Order_ID','count'))seg

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,5))sns.barplot(x=seg.index, y=seg['Total_Sales'], hue=seg.index, palette='mako', ax=axes[0], legend=False)axes[0].set_title('Total Sales by Customer Segment', fontweight='bold')sns.barplot(x=seg.index, y=seg['Avg_Order'], hue=seg.index, palette='mako', ax=axes[1], legend=False)axes[1].set_title('Average Order Value by Segment', fontweight='bold')plt.tight_layout(); plt.show()

### 5.6 Payment Method Analysis

In [ ]:
pay = df['Payment_Method'].value_counts()plt.figure(figsize=(6.5,6.5))colors = sns.color_palette('Set2', len(pay))plt.pie(pay.values, labels=pay.index, autopct='%1.1f%%', colors=colors, startangle=90,        wedgeprops={'edgecolor':'white','linewidth':1.5})plt.title('Payment Method Distribution', fontsize=14, fontweight='bold')plt.tight_layout(); plt.show()

### 5.7 Revenue & Profit Overview

In [ ]:
yearly = df.groupby('Year').agg(Total_Sales=('Total_Sales','sum'), Profit=('Profit','sum'), Orders=('Order_ID','count'))yearly

In [ ]:
x = np.arange(len(yearly)); width = 0.38plt.figure(figsize=(8,5))plt.bar(x-width/2, yearly['Total_Sales'], width, label='Revenue', color='#264653')plt.bar(x+width/2, yearly['Profit'], width, label='Profit', color='#e76f51')plt.xticks(x, yearly.index)plt.title('Yearly Revenue vs Profit', fontsize=14, fontweight='bold')plt.legend(); plt.tight_layout(); plt.show()

### 5.8 Correlation Heatmap

In [ ]:
numeric_cols = ['Quantity','Unit_Price','Discount_Percent','Total_Sales','Shipping_Cost','Profit']plt.figure(figsize=(7,6))sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')plt.tight_layout(); plt.show()

## 6. Machine Learning — Sales PredictionGoal: predict `Total_Sales` for an order from order/customer/product attributes.We prepare features, encode categoricals, scale numerics, split into train/test sets, and train three regression models.

In [ ]:
features = ['Customer_Segment','Country','Region','Product_Category','Quantity',            'Unit_Price','Discount_Percent','Shipping_Cost','Month','Quarter','DayOfWeek']target = 'Total_Sales'X = df[features]y = df[target]categorical = ['Customer_Segment','Country','Region','Product_Category','DayOfWeek']numeric = ['Quantity','Unit_Price','Discount_Percent','Shipping_Cost','Month','Quarter']preprocessor = ColumnTransformer(transformers=[    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),    ('num', StandardScaler(), numeric)])X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)print('Train size:', X_train.shape, ' Test size:', X_test.shape)

### 6.1 Train & Compare Models

In [ ]:
models = {    'Linear Regression': LinearRegression(),    'Decision Tree Regressor': DecisionTreeRegressor(random_state=42, max_depth=8),    'Random Forest Regressor': RandomForestRegressor(random_state=42, n_estimators=200, max_depth=12)}results = {}pipelines = {}for name, model in models.items():    pipe = Pipeline([('prep', preprocessor), ('model', model)])    pipe.fit(X_train, y_train)    preds = pipe.predict(X_test)    mae = mean_absolute_error(y_test, preds)    rmse = np.sqrt(mean_squared_error(y_test, preds))    r2 = r2_score(y_test, preds)    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}    pipelines[name] = pipe    print(f'{name}: MAE={mae:.2f}, RMSE={rmse:.2f}, R2={r2:.4f}')

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)results_df

### 6.2 Select Best Model

In [ ]:
best_model_name = results_df.index[0]best_pipe = pipelines[best_model_name]print('Best model:', best_model_name)

In [ ]:
results_df['R2'].sort_values().plot(kind='barh', color='#2a9d8f', figsize=(8,5))plt.title('Model Comparison — R² Score', fontsize=14, fontweight='bold')plt.xlabel('R² Score')plt.tight_layout(); plt.show()

In [ ]:
best_preds = best_pipe.predict(X_test)plt.figure(figsize=(7,6))plt.scatter(y_test, best_preds, alpha=0.4, color='#264653')lims = [0, max(y_test.max(), best_preds.max())]plt.plot(lims, lims, 'r--', linewidth=1.5)plt.xlabel('Actual Sales ($)'); plt.ylabel('Predicted Sales ($)')plt.title(f'Actual vs Predicted — {best_model_name}', fontsize=13, fontweight='bold')plt.tight_layout(); plt.show()

### 6.3 Save the Trained Model

In [ ]:
with open('sales_prediction_model.pkl', 'wb') as f:    pickle.dump({        'model': best_pipe,        'features': features,        'categorical': categorical,        'numeric': numeric,        'model_name': best_model_name,        'metrics': results    }, f)print('Model saved to sales_prediction_model.pkl')

## 7. Key Business Insights- **Revenue & Profit**: Total revenue and profit trends are broken down by year and month above.- **Top Category**: Furniture leads in total revenue.- **Top Region**: Europe generates the highest sales.- **Top Payment Method**: Credit Card is the most used method (~40% of orders).- **Best Model**: Random Forest Regressor achieved the highest R² and lowest error, making it the most reliable model for predicting future sales.These insights, along with an interactive prediction tool, are available in the accompanying Streamlit dashboard (`app.py`).